In [1]:
import torch
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.multiprocessing import Process, Queue
from torch.multiprocessing import set_start_method
from transformers import AutoImageProcessor, SwinForImageClassification
import os
import sys
from tqdm import tqdm
sys.path.append('/work/k-kuroki/matrix_factorization/quantizer')
import quantizer as qtz
from quantizer import QuadraticQuantization as QQ, BinaryCodingQuantization as BCQ, UniformQuantization as UQ, LatticeVectorQuantization as LVQ, BinaryQuadraticQuantization as BQQ, BinaryMatrixFactorization as BMF
import queue
from multiprocessing import Process, Queue, current_process
import pandas as pd
import numpy as np
from transformers import AutoImageProcessor, SwinForImageClassification, DeiTForImageClassificationWithTeacher, DeiTForImageClassification
import timm
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
sys.path.append('/work2/k-kuroki/quadratic_quantization/quantizer')
sys.path.append('/work2/k-kuroki/quadratic_quantization/nn_compression/cv')
sys.path.append('/work/k-kuroki/gpu_from_python')
from build_dataset import get_imagenet
from integrated_generator import TSP

import torch._dynamo
torch._dynamo.config.suppress_errors = True

# バイナリファイルからロード
def fvecs_read(file_name):
    with open(file_name, 'rb') as f:
        data = np.fromfile(f, dtype=np.float32)
        d = data.view(np.int32)[0]  # 次元数
        return data.reshape(-1, d + 1)[:, 1:]
    
def get_matrix_data():
    torch.manual_seed(1)
    # 実験用行列
    w1 = torch.randn(128, 128)
    print('Random data', w1.shape)

    # DeiT-S重み
    model = timm.create_model("deit_small_patch16_224", pretrained=True)
    model_weights = model.state_dict()
    w_name = ['intermediate', 'key', 'value', 'dense', 'query']
    for i, (name, param) in enumerate(model_weights.items()):
        # print(f"Layer: {name} - Shape: {param.shape}")
        w2 = param.data
        if i==8:
            # print(f"Layer: {name} - Shape: {param.shape}")
            break
    print('DeiT-S data', w2.shape)


    # TSPLIB距離グラフ
    problem_name = 'rd100'
    problem_path = r'/work/k-kuroki/problem/tsp/{0}/{0}.tsp'.format(problem_name)
    instance = TSP(problem_path)
    w3 = torch.tensor(instance.Dij)
    print('TSP data', w3.shape)

    # ANNデータ
    dataname_list = ['siftsmall', 'sift']
    data_name = dataname_list[1]
    w4 = torch.tensor(fvecs_read('/work2/k-kuroki/quadratic_quantization/ann/dataset/{0}/{0}_base.fvecs'.format(data_name))[:128])
    print('ANN data', w4.shape)


    # ImageNet
    data_path = '/ldisk/Shared/Datasets/ILSVRC/ILSVRC2012/'
    train, val = get_imagenet(model_name='deit', num_traindatas=32)
    for batch_idx, (images, _) in enumerate(train):
        w5 = images[5][0]
        break
    print('ImageNet data', w5.shape)
    return [w1, w2, w3, w4, w5]



BCQの動作確認

In [ ]:
torch.manual_seed(1)

matrix_data = get_matrix_data()
input = matrix_data[4]#[:, :128]
input = torch.randn(128, 128)
print('Input shape:', input.shape)
# W = matrix_data[1]
W = torch.randn(input.shape[1], input.shape[1])

def f(x, y, z, a, s=input.T@input/input.shape[0]):
    q = (a[:,0].unsqueeze(-1).unsqueeze(-1)*y@z + a[:,1].unsqueeze(-1).unsqueeze(-1)*y.sum(dim=-1, keepdim=True) + a[:,2].unsqueeze(-1).unsqueeze(-1)*z.sum(dim=-2, keepdim=True) + a[:,3].unsqueeze(-1).unsqueeze(-1)).sum(dim=0)
    return torch.trace((x - q) @ s @ (x - q).T)

Wbcq, B, alpha = BCQ().run_bcq(w=W.clone(), bit_width=1, Nstep=100)
Wbcq = Wbcq.cpu()
# print('BCQ error:',((W - Wbcq.cpu())**2).mean())
print('BCQ activation error:',((input@(W - Wbcq.cpu()).T)**2).mean())
print('cos sim',(W * (Wbcq)).sum() / (W.norm() * (Wbcq).norm()))

Wuq = UQ().run_uq(W.clone(), 1, device=torch.device('cuda:2'))
Wuq = Wuq.cpu()
print('UQ activation error:',((input@(W - Wuq.cpu()).T)**2).mean())
print('cos sim',(W * (Wuq)).sum() / (W.norm() * (Wuq).norm()))


# y, z, a = BQQ().run_bqq_multibit(W.clone(), bit_width=1, rank_scale=1, device_id=0, Tinit=0.2, Tfin=0.005, Nstep=100000, eta=0.06, zeta=4, damping=0.005)
# y, z, a = y.cpu(), z.cpu(), a.cpu()
# Wbqq = (a[:,0].unsqueeze(-1).unsqueeze(-1)*y@z + a[:,1].unsqueeze(-1).unsqueeze(-1)*y.sum(dim=-1, keepdim=True) + a[:,2].unsqueeze(-1).unsqueeze(-1)*z.sum(dim=-2, keepdim=True) + a[:,3].unsqueeze(-1).unsqueeze(-1)).sum(dim=0)
# print('fast BQQ error:',((W - Wbqq.cpu())**2).mean())
# print('fast BQQ activation error:',((input@(W - Wbqq.cpu()).T)**2).mean())
# print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))

# SIFTの時：Tinit=3e-3, Tfin=-3e-3, Nstep=50000, eta=0.1, zeta=30, damping=0, clipping=0.1 / RANDOMの時:  Tinit=5e-3, Tfin=-3e-3, Nstep=50000, eta=0.1, zeta=30, damping=0, clipping=0.5 / ImageNet : Tinit=5e-3, Tfin=-3e-3, Nstep=30000, eta=0.2, zeta=20, damping=1e-15, clipping=0.5
y, z, a = BQQ().run_activation_aware_bqq_multibit(W.clone(), input=input, bit_width=1, rank_scale=1, device_id=0, Tinit=5e-3, Tfin=-3e-3, Nstep=50000, eta=0.1, zeta=30, damping=0, clipping=0.5)
y, z, a = y.cpu(), z.cpu(), a.cpu()
Wbqq = (a[:,0].unsqueeze(-1).unsqueeze(-1)*y@z + a[:,1].unsqueeze(-1).unsqueeze(-1)*y.sum(dim=-1, keepdim=True) + a[:,2].unsqueeze(-1).unsqueeze(-1)*z.sum(dim=-2, keepdim=True) + a[:,3].unsqueeze(-1).unsqueeze(-1)).sum(dim=0)
print('fast activation BQQ error:',((input@(W - Wbqq.cpu()).T)**2).mean())
print('fast activation BQQ error:',(f(W, y, z, a))/W.shape[0])
print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))



# Wbqq = BQQ().bqq_large_matrix_multi_worker(W, rank_scale=1, bit_width=2, max_patch_size=128, Nstep=40000, Tinit=0.2, Tfin=0.005, eta=0.06, zeta=4)
# print('BQQ error:',((W - Wbqq.cpu())**2).mean())
# print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))

# Wbqq = QQ(W, rank_scale=1).qq_large_matrix_multi_worker(bit_width=2, max_patch_size=128, Nstep=20000, Tinit=0.2, Tfin=0.005, eta=0.06, zeta=4)
# print('BQQ error:',((W - Wbqq.cpu())**2).mean())
# print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))

# y, z, a = BQQ().run_bqq_compile(W, rank_scale=2, device_id=0, Nstep=20000, Tinit=0.2, Tfin=0.005, eta=0.06, zeta=4)
# y, z, a = y.cpu(), z.cpu(), a.cpu()
# Wbqq = a[0] * y@z + a[1] * y.sum(axis=1).unsqueeze(1) + a[2] * z.sum(axis=0).unsqueeze(0) + a[3]
# print('BQQ error:',((W - Wbqq.cpu())**2).mean())
# print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))

# y, z, a = QQ(W, rank_scale=2).run_qq_compile(device_id=0, Nstep=20000, Tinit=0.2, Tfin=0.005, eta=0.06, zeta=4, output_type='torch')
# y, z, a = y.cpu(), z.cpu(), a.cpu()
# Wbqq = a[0] * y@z + a[1] * y.sum(axis=1).unsqueeze(1) + a[2] * z.sum(axis=0).unsqueeze(0) + a[3]
# print('BQQ error:',((W - Wbqq.cpu())**2).mean())
# print('cos sim', (W * (Wbqq)).sum() / (W.norm() * (Wbqq).norm()))

# y, z, a = BMF().run_binary_multi(W-W.min(), rank_scale=1, device_id=0, Nstep=50000, zeta=4, eta=0.04, Tinit=0.2, Tfin=0.05)
# y, z, a = y.cpu(), z.cpu(), a.cpu()
# Wbmf = (a * y @ z).squeeze(0) + W.min()
# print('BMF error:',((W - Wbmf.cpu())**2).mean())
# print('cos sim', (W * (Wbmf)).sum() / (W.norm() * (Wbmf).norm()))

Random data torch.Size([128, 128])
DeiT-S data torch.Size([384, 384])
TSP data torch.Size([100, 100])
ANN data torch.Size([128, 128])
ImageNet data torch.Size([224, 224])
Input shape: torch.Size([128, 128])
BCQ activation error: tensor(46.4096)
cos sim tensor(0.7967)
UQ activation error: tensor(46.4434)
cos sim tensor(0.7967)


  1%|          | 493/50000 [00:00<00:10, 4922.58it/s]

(y,z): 0.053511910140514374 0.05394294112920761
energy: 3.846479892730713


  4%|▍         | 1994/50000 [00:00<00:09, 4804.76it/s]

(y,z): 0.0946483165025711 0.09598587453365326
energy: 2.4879658222198486


...

 64%|██████▍   | 31908/50000 [00:06<00:03, 4955.24it/s]

(y,z): 0.21255485713481903 0.1978943645954132
energy: 0.7228565216064453
(y,z): 0.219824880361557 0.2026674449443817


 66%|██████▌   | 32908/50000 [00:06<00:03, 4979.57it/s]

energy: 0.7042754888534546
(y,z): 0.2246609926223755 0.20638641715049744
energy: 0.6923302412033081


 69%|██████▉   | 34404/50000 [00:06<00:03, 4971.59it/s]

QQとSOAQが等価だけど最適化の種類でどれくらい誤差が変わるのかの検証

In [ ]:
W = torch.randn(64, 64)
W = torch.exp(-torch.abs(W))
ist = QQ(x=W, rank_scale=1)
# 最適パラメータ設定 (次元が大きくなるにつれて最終温度を下げないといけなくなるかも)
(4, 0.06, 0.2, 0.002) #0.3537
y, z, a = ist.run_qq_compile(zeta=1, eta=0.1, Tinit=0.2, Tfin=0.02, Nstep=20000, device_id=0, seed=1, output_type='torch')
W_qq = a[0]*y@z + a[1]*y@torch.ones(z.shape, device=y.device) + a[2]*torch.ones(y.shape, device=z.device)@z + a[3]
print(((W - W_qq.cpu())**2).mean())
ist.x = W.unsqueeze(0)
Wq = ist.qq_large_matrix_multi_worker(max_patch_size=192, bit_width=1, save_name=None, zeta=4, eta=0.06, Tinit=0.2, Tfin=0.005, Nstep=20000, seed=1, workers_per_gpu=16)
print(((W-Wq)**2).mean())

QQの最適化の中で使われる勾配とヘッセ行列が正しく計算できているかの確認コード（自動計算と比べる）

In [ ]:
import time

def compute_H(x, y, z, a):
    yz = y @ z
    sigma_y = y.sum(axis=1, keepdim=True)
    sigma_z = z.sum(axis=0, keepdim=True)
    
    part1 = (x - (a[0] * yz + a[1] * sigma_y + a[2] * sigma_z + a[3])) ** 2
    part2 = a[0]**2 * (yz - (y**2) @ (z**2))
    part3 = a[1]**2 * (sigma_y - (y**2).sum(axis=1, keepdim=True))
    part4 = a[2]**2 * (sigma_z - (z**2).sum(axis=0, keepdim=True))
    part5 = 2*a[0]*a[1] * (yz - y**2 @ z)
    part6 = 2*a[0]*a[2] * (yz - y @ z**2)
    
    return (part1 + part2 + part3 + part4 + part5 + part6).sum()

def compute_grads(x, y, z, a):
    n, m = x.shape
    # パーツの計算
    yz, sigma_y, sigma_z = y @ z, y.sum(axis=1).unsqueeze(1), z.sum(axis=0).unsqueeze(0)
    part = x - (a[3] + a[0] * yz + a[1] * sigma_y + a[2] * sigma_z) ## sigma_zはaxis=1でsumだけどここではaxis=0でsumなことに注意 (yも同様)

    # 平均場エネルギー勾配の計算
    # y_grad = (-2 * part @ (a[0] * z + a[1]).T) + (a[0]**2) * (z.sum(axis=1).unsqueeze(0) - 2 * y * (z**2).sum(axis=1).unsqueeze(0)) + (a[1]**2) * (1 - 2 * y) * m
    # z_grad = (-2 * (a[0] * y + a[2]).T @ part) + (a[0]**2) * (y.sum(axis=0).unsqueeze(1) - 2 * z * (y**2).sum(axis=0).unsqueeze(1)) + (a[2]**2) * (1 - 2 * z) * n

    # 平均場エネルギー勾配の計算
    y_grad = (-2 * part @ (a[0] * z + a[1]).T) + (a[0]**2 + 2*a[0]*a[1]*(1 - 2*y) + 2*a[0]*a[2]) * (z.sum(axis=1).unsqueeze(0)) - 2 * (a[0]*a[2] + a[0]**2 * y) * (z**2).sum(axis=1).unsqueeze(0) + (a[1]**2) * (1 - 2 * y) * m
    z_grad = (-2 * (a[0] * y + a[2]).T @ part) + (a[0]**2 + 2*a[0]*a[1] + 2*a[0]*a[2]*(1 - 2*z)) * (y.sum(axis=0).unsqueeze(1)) - 2 * (a[0]**2 * z + a[0]*a[1]) * (y**2).sum(axis=0).unsqueeze(1) + (a[2]**2) * (1 - 2 * z) * n

    return y_grad, z_grad



# Define tensor sizes
# torch.manual_seed(1)
n, r, m = 100, 80, 90
y = torch.rand((n, r), requires_grad=True, dtype=torch.float64)
z = torch.rand((r, m), requires_grad=True, dtype=torch.float64)
a = torch.rand(4, requires_grad=True, dtype=torch.float64)
x = torch.randn(n, m, dtype=torch.float64)

st = time.time()
# Compute function H
H = compute_H(x, y, z, a)
# Compute gradients using autograd
H.backward()
end = time.time()
print('auto grad time:', end - st)

y_grad_autograd = y.grad.clone()
z_grad_autograd = z.grad.clone()

# Compute given gradients
st = time.time()
y_grad_manual, z_grad_manual = compute_grads(x, y, z, a)
end = time.time()
print('manual grad time:', end-st)

# Compare results
y_match = torch.allclose(y_grad_autograd, y_grad_manual, atol=1e-8)
z_match = torch.allclose(z_grad_autograd, z_grad_manual, atol=1e-5)

# print(y_grad_autograd - y_grad_manual)
# print(z_grad_autograd - z_grad_manual)
print("y gradient match:", y_match)
print("z gradient match:", z_match)

def hessian_H(x, y, z, a):
    return torch.autograd.functional.hessian(lambda a: compute_H(x, y, z, a), a)

def compute_hesse(x, y, z):
    # パーツの計算
    coeff = -2 * x.sum()
    yz, y2z2, sigma_y, sigma_y2, sigma_z, sigma_z2 = y @ z, y**2 @ z**2, y.sum(axis=1).unsqueeze(1), (y**2).sum(axis=1).unsqueeze(1), z.sum(axis=0).unsqueeze(0), (z**2).sum(axis=0).unsqueeze(0) 
    # スケーリング係数の最適化
    r0c0, r0c1, r0c2, r0c3, r1c1, r1c2, r1c3, r2c2, r2c3, r3c3 = (yz**2 + yz - y2z2).sum(), ((sigma_y + 1) * yz - y**2 @ z).sum(), ((1 + sigma_z) * yz - y @ z**2).sum(), yz.sum(), (sigma_y**2 + sigma_y - sigma_y2).sum() * m, (sigma_y * sigma_z).sum(), sigma_y.sum() * m, (sigma_z**2 + sigma_z - sigma_z2).sum() * n, sigma_z.sum() * n, n * m
    hesse = 2*torch.tensor([[r0c0, r0c1, r0c2, r0c3],
                [r0c1, r1c1, r1c2, r1c3],
                [r0c2, r1c2, r2c2, r2c3],
                [r0c3, r1c3, r2c3, r3c3]])
    v = torch.tensor([(-2 * x * yz).sum(), (-2 * x * sigma_y).sum(), (-2 * x * sigma_z).sum(), coeff])
    return hesse, v

# Compute given hesse
st = time.time()
for i in range(100):
    hesse_auto = hessian_H(x, y, z, a)
en = time.time()
print('auto hesse time:', en - st)

st = time.time()
for i in range(100):
    hesse_manual, v = compute_hesse(x, y, z)
en = time.time()
print('manual hesse time:', en - st)

hesse_match = torch.allclose(hesse_auto, hesse_manual, atol=1e-8)
print('hesse match:', hesse_match)

# ヘッセ行列の固有値
print('hesse eigvals are positive:', torch.where(torch.linalg.eigvalsh(hesse_manual)>0, True, False))
a_new1 = -v @ torch.linalg.inv(hesse_manual)
a_new2 = a - a.grad @ torch.linalg.inv(hesse_manual)
print('newton match:', torch.allclose(a_new1, a_new2, atol=1e-8))



SOAQをQQの形にした時に目的関数の値が一致するかどうか確認するコード

In [ ]:
# SOAQの目的関数
def soaq_energy(x, y, z, a):
    yz = y@z
    y_z = (1-y)@z
    yz_ = y@(1-z)
    y_z_ = (1-y)@(1-z)
    reconst = a[0]*yz + a[1]*y_z + a[2]*yz_ + a[3]*y_z_
    binarize_term = ((a[0])**2)*(yz - (y**2)@(z**2)) + ((a[1])**2)*(y_z - ((1-y)**2)@(z**2)) + ((a[2])**2)*(yz_ - (y**2)@((1-z)**2)) + ((a[3])**2)*(y_z_ - ((1-y)**2)@((1-z)**2))
    # reconst = (a[0]*y + a[1]*(1-y))@z + (a[2]*y+a[3]*(1-y))@(1-z) 
    return ((x - reconst)**2 + binarize_term).sum() 
    return ((x - reconst)**2).sum()

# QQの目的関数
def qq_energy(x, y, z, a):
    yz = y @ z
    sigma_y = y.sum(axis=1, keepdim=True)
    sigma_z = z.sum(axis=0, keepdim=True)
    
    part1 = (x - (a[0] * yz + a[1] * sigma_y + a[2] * sigma_z + a[3])) ** 2
    part2 = a[0]**2 * (yz - (y**2) @ (z**2))
    part3 = a[1]**2 * (sigma_y - (y**2).sum(axis=1, keepdim=True))
    part4 = a[2]**2 * (sigma_z - (z**2).sum(axis=0, keepdim=True))
    part5 = 2*a[0]*a[1] * (yz - y**2 @ z)
    part6 = 2*a[1]*a[2] * (yz - y @ z**2)

    return (part1 + part2 + part3 + part4 + part5 + part6).sum()
    # return part1.sum()

# 確認
# torch.manual_seed(0)
n, r, m = 100, 80, 90
y = torch.rand((n, r), requires_grad=True, dtype=torch.float64)
# y = (torch.where(y>0.5, 1.0, 0.0)).to(torch.float64)
z = torch.rand((r, m), requires_grad=True, dtype=torch.float64)
# z = (torch.where(z>0.5, 1.0, 0.0)).to(torch.float64)
a = torch.rand(4, requires_grad=True, dtype=torch.float64)
x = torch.randn(n, m, dtype=torch.float64)
soaq = soaq_energy(x, y, z, a)

# 係数変換
e = torch.tensor([
    a[0] - a[1] - a[2] + a[3],  # e0
    a[2] - a[3],                # e1
    a[1] - a[3],                # e2
    r*a[3]                         # e3
], dtype=torch.float64)
qq = qq_energy(x, y, z, e)

print(qq - soaq, qq==soaq)


３D行列を２D行列に変換して、また３Dに戻したときに元の行列と合っているかどうかの確認

In [ ]:
def aggregate_matrices(patch_list, output_shape, bit_width, qtz_type='qq'):
    result_matrix = torch.zeros(output_shape)
    # 特定のビット重みのみ
    for patch in patch_list:
        i, j = patch['patch_row'], patch['patch_col']
        a, y, z, k = patch['coeff'], patch['mat1'], patch['mat2'], patch['bit_idx']
        
        if k >= bit_width:
            continue  # bit_idx が範囲外の場合はスキップ
        
        if qtz_type == 'soaq':
            term1 = (a[0] * y + a[1] * (1 - y)) @ z
            term2 = (a[2] * y + a[3] * (1 - y)) @ (1 - z)
            patch_result = term1 + term2
        elif qtz_type == 'qq':
            patch_result = a[0]*y@z + a[1]*y.sum(axis=1, keepdim=True) + a[2]*z.sum(axis=0, keepdim=True) + a[3]

        
        # 結果を i, j の位置に格納
        row_start, row_end = i * patch_result.shape[0], (i + 1) * patch_result.shape[0]
        col_start, col_end = j * patch_result.shape[1], (j + 1) * patch_result.shape[1]
        result_matrix[row_start:row_end, col_start:col_end] += patch_result

    return result_matrix

def confirm(param):
    original_shape = param.shape
    conversion_shape = param.reshape(param.shape[0], -1).shape
    print('original shape:{0}, conversion shape:{1}'.format(original_shape, conversion_shape))
    reconst1 = QQ(x=param.reshape(param.shape[0], -1).unsqueeze(0), rank_scale=1).qq_large_matrix_multi_worker(max_patch_size=48, bit_width=1, Nstep=50000, save_name=r'/work/k-kuroki/matrix_factorization/nn_compression/test_result/test')
    reconst1_original_shape = reconst1.reshape(original_shape)

    weight_list = []
    weights_dir = r'/work/k-kuroki/matrix_factorization/nn_compression/test_result'
    # ディレクトリにあるファイルを逐次的に読み込み
    for file in os.listdir(weights_dir):
        # 層の名前が読み込むファイルと一致している場合に処理を行う(ファイル名_row{i}_col{j}.pthとなっているものだけを対象とする)
        if file.endswith(".pth"): # アテンションの重みをロードする
            weight_path = os.path.join(weights_dir, file) # フルパスを作成(fileはfile名のみ)
            weight_list += torch.load(weight_path, weights_only=False, map_location=torch.device('cpu'))
    
    # 保存したものを読み込む
    reconst2 = aggregate_matrices(weight_list, conversion_shape, bit_width=1, qtz_type='qq')
    reconst2_original_shape = reconst2.reshape(original_shape)


    print('分解行列が保存前後で合っているか？', torch.allclose(reconst1, reconst2, 1e-5), torch.allclose(reconst1_original_shape, reconst2_original_shape, 1e-5))
    print('誤差が一致しているか？', torch.allclose(torch.mean((param - reconst1_original_shape)**2), torch.mean((param - reconst2_original_shape)**2), 1e-5))
    print('分解前後の差', reconst1-reconst2)
    print('平均誤差の値x100', 100*torch.mean((param - reconst2_original_shape)**2))



def load(model, extract='embedding'):
    if extract == 'embedding':
        emb = True
        head = False
    elif extract == 'head':
        emb = False
        head = True
    # 元モデルのロード
    for name, param in model.named_parameters():

        if 'weight' in name and ('emb' in name) and not 'norm' in name and emb:
            print(name, param.shape)
            return param
        elif 'weight' in name and ('head' in name or 'classifier' in name) and not 'norm' in name and head:
            print(name, param.shape)
            return param           

def get_model(model_abbreviation='swin-s'):
    if model_abbreviation == 'swin-s':
        model_name = "microsoft/swin-small-patch4-window7-224"
        model = SwinForImageClassification.from_pretrained(model_name)
        # model = timm.create_model("swin_small_patch4_window7_224", pretrained=True)

    if model_abbreviation == 'swin-t':
        model_name = "microsoft/swin-tiny-patch4-window7-224"
        model = SwinForImageClassification.from_pretrained(model_name)
        # model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True)

    if model_abbreviation == 'resnet18':
        model = models.resnet18(weights='DEFAULT')
        checkpoint_path = "/work/k-kuroki/resnet18_imagenet.pth.tar"  # 実際のパスに変更
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(state_dict)

    if model_abbreviation == 'resnet50':
        model = models.resnet50(weights='DEFAULT')
        checkpoint_path = "/work/k-kuroki/resnet50_imagenet.pth.tar"  # 実際のパスに変更
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(state_dict)

    if model_abbreviation == 'deit-b':
        # model_name = 'facebook/deit-base-patch16-224' # DeiT-B
        # model = DeiTForImageClassification.from_pretrained(model_name, ignore_mismatched_sizes=False)
        model = timm.create_model("deit_base_patch16_224", pretrained=True)

    if model_abbreviation == 'deit-s':
        # model_name = 'facebook/deit-small-patch16-224' # DeiT-S
        # model = DeiTForImageClassification.from_pretrained(model_name, ignore_mismatched_sizes=False)
        model = timm.create_model("deit_small_patch16_224", pretrained=True)

    return model

def hist_matrix(matrix, bins=20):
    """
    行列 (2次元) またはカラーチャネルを持つ行列 (3次元) の各要素のヒストグラムを横に並べてプロットします.

    Args:
        matrix: ヒストグラムを作成する行列 (2次元) またはカラーチャネルを持つ行列 (3次元)
        bins: ヒストグラムのビンの数
    """

    if matrix.ndim > 2:
        num_colors = matrix.shape[0]
        cmap = ['r', 'g', 'b']

        # サブプロットを生成
        fig, axes = plt.subplots(1, num_colors, figsize=(15, 5))

        for color_idx, ax in enumerate(axes):
            chunnel_matrix = matrix[color_idx]
            channel_data = chunnel_matrix.ravel()
            print(f"Channel {color_idx+1} data range: {channel_data.min()} to {channel_data.max()}")
            # hist, bin_edges = np.histogram(chunnel_matrix.ravel(), bins=bins)
            # ax.bar(bin_edges[:-1], hist, color=cmap(color_idx), label=f"Channel {color_idx+1}")
            ax.hist(channel_data, bins=bins, color=cmap[0])
            ax.set_xlabel('Value')
            ax.set_ylabel('Frequency')
            ax.legend()

        plt.tight_layout()
        plt.show()
    else:
        # 単一のヒストグラムをプロット
        plt.hist(matrix.ravel(), bins=bins, color='navy')
        plt.xlabel('Value')
        plt.ylabel('Frequency')
        plt.show()


# torch.manual_seed(0)
# param = torch.randn((96, 3, 4, 4))

model = get_model('swin-t')
param = load(model=model, extract='embedding')
hist_matrix((param.reshape(param.shape[0], -1)).detach().numpy())
confirm(param.detach())


Llamaモデルの読み込みができるかどうか

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-2-7b-hf"  # 例: LLaMA 2-7B
# model_name = "meta-llama/Meta-Llama-3-8B"  # 例: LLaMA 3-8B (LLaMA 3 の場合)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# テスト
text = "Hello, how are you?"
inputs = tokenizer(text, return_tensors="pt").to("cuda")
output = model.generate(**inputs)
print(tokenizer.decode(output[0]))
